### NECESSARY LIBRARIES

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
plt.style.use('seaborn-v0_8-darkgrid')

print("="*70)
print("TASK 2: MODEL BUILDING AND TRAINING")
print("="*70)

### 1. LOAD PROCESSED DATA

In [ ]:
print("\n1. Loading processed data...")

# Load engineered data from Task 1
fraud_data = pd.read_csv('data/processed/fraud_data_engineered.csv')
credit_data = pd.read_csv('data/processed/creditcard_processed.csv')

print(f"Fraud Data shape: {fraud_data.shape}")
print(f"Credit Data shape: {credit_data.shape}")

### 2. PREPARE E-COMMERCE DATA FOR MODELING

In [ ]:
print("\n" + "="*70)
print("2. PREPARING E-COMMERCE DATA")
print("="*70)

from src.model_training import FraudDetectionModel

# Select features for e-commerce modeling
# We'll exclude identifiers and timestamps
excluded_features = ['user_id', 'signup_time', 'purchase_time', 'ip_address']

# Separate features and target
X_ecomm = fraud_data.drop(['class'] + excluded_features, axis=1, errors='ignore')
y_ecomm = fraud_data['class']

# Handle categorical features
categorical_cols = X_ecomm.select_dtypes(include=['object', 'category']).columns
print(f"Categorical columns: {list(categorical_cols)}")

# One-hot encode categorical variables
X_ecomm_encoded = pd.get_dummies(X_ecomm, columns=categorical_cols, drop_first=True)

print(f"\nFeatures after encoding: {X_ecomm_encoded.shape[1]}")
print(f"Target distribution: {np.unique(y_ecomm, return_counts=True)}")

### 3. TRAIN MODELS FOR E-COMMERCE DATA

In [ ]:
print("\n" + "="*70)
print("3. TRAINING MODELS (E-COMMERCE)")
print("="*70)

# Initialize model trainer
model_trainer = FraudDetectionModel(random_state=42)

# Prepare data with SMOTE
X_train, X_test, y_train, y_test = model_trainer.prepare_data(
    X_ecomm_encoded, y_ecomm, 
    test_size=0.2, 
    apply_smote=True
)

# Train baseline Logistic Regression
logistic_model, logistic_results = model_trainer.train_baseline_logistic(
    X_train, X_test, y_train, y_test,
    C=0.1,  # Regularization parameter
    class_weight='balanced'
)

# Train Random Forest
rf_model, rf_results = model_trainer.train_random_forest(
    X_train, X_test, y_train, y_test,
    n_estimators=150,
    max_depth=15,
    min_samples_leaf=3
)

# Train XGBoost
xgb_model, xgb_results = model_trainer.train_xgboost(
    X_train, X_test, y_train, y_test,
    n_estimators=150,
    max_depth=8,
    learning_rate=0.05
)

### 4. CROSS-VALIDATION

In [ ]:
print("\n" + "="*70)
print("4. CROSS-VALIDATION ANALYSIS")
print("="*70)

# Perform cross-validation for each model
cv_results = {}

# Logistic Regression CV
cv_logistic = model_trainer.perform_cross_validation(
    LogisticRegression(C=0.1, class_weight='balanced', random_state=42),
    X_ecomm_encoded, y_ecomm,
    cv=5,
    model_name="Logistic Regression"
)
cv_results['logistic'] = cv_logistic

# Random Forest CV
cv_rf = model_trainer.perform_cross_validation(
    RandomForestClassifier(n_estimators=150, max_depth=15, 
                          min_samples_leaf=3, random_state=42,
                          class_weight='balanced_subsample', n_jobs=-1),
    X_ecomm_encoded, y_ecomm,
    cv=5,
    model_name="Random Forest"
)
cv_results['random_forest'] = cv_rf

# XGBoost CV
cv_xgb = model_trainer.perform_cross_validation(
    XGBClassifier(n_estimators=150, max_depth=8, learning_rate=0.05,
                  scale_pos_weight=len(y_ecomm[y_ecomm==0])/len(y_ecomm[y_ecomm==1]),
                  random_state=42, eval_metric='aucpr', use_label_encoder=False),
    X_ecomm_encoded, y_ecomm,
    cv=5,
    model_name="XGBoost"
)
cv_results['xgboost'] = cv_xgb

### 5. MODEL COMPARISON AND SELECTION

In [ ]:
print("\n" + "="*70)
print("5. MODEL COMPARISON")
print("="*70)

# Compare all models
comparison_df = model_trainer.compare_models()

# Additional comparison with CV results
print("\n" + "-"*40)
print("CROSS-VALIDATION COMPARISON")
print("-"*40)

cv_comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'CV AUC Mean': [cv_results['logistic']['auc_mean'], 
                   cv_results['random_forest']['auc_mean'], 
                   cv_results['xgboost']['auc_mean']],
    'CV AUC Std': [cv_results['logistic']['auc_std'], 
                  cv_results['random_forest']['auc_std'], 
                  cv_results['xgboost']['auc_std']],
    'CV F1 Mean': [cv_results['logistic']['f1_mean'], 
                  cv_results['random_forest']['f1_mean'], 
                  cv_results['xgboost']['f1_mean']],
    'CV F1 Std': [cv_results['logistic']['f1_std'], 
                 cv_results['random_forest']['f1_std'], 
                 cv_results['xgboost']['f1_std']]
})

print(cv_comparison.round(4))

### 6. SAVE MODELS

In [ ]:
print("\n" + "="*70)
print("6. SAVING MODELS")
print("="*70)

# Save all models
model_trainer.save_model(logistic_model, 'logistic_regression_ecomm')
model_trainer.save_model(rf_model, 'random_forest_ecomm')
model_trainer.save_model(xgb_model, 'xgboost_ecomm')

# Save feature names for later use
feature_names = X_ecomm_encoded.columns.tolist()
joblib.dump(feature_names, 'models/feature_names_ecomm.pkl')
print(f"Feature names saved: {len(feature_names)} features")

### 7. CREDIT CARD DATA MODELING

In [ ]:
print("\n" + "="*70)
print("7. CREDIT CARD DATA MODELING")
print("="*70)

# Prepare credit card data
X_credit = credit_data.drop(['Class', 'Time', 'Time_hours', 'Time_bin'], axis=1, errors='ignore')
y_credit = credit_data['Class']

print(f"Credit features: {X_credit.shape[1]}")
print(f"Credit target distribution: {np.unique(y_credit, return_counts=True)}")

# Train models for credit data
model_trainer_credit = FraudDetectionModel(random_state=42)

# Prepare data
X_train_c, X_test_c, y_train_c, y_test_c = model_trainer_credit.prepare_data(
    X_credit, y_credit, test_size=0.2, apply_smote=True
)

# Train models
logistic_credit, _ = model_trainer_credit.train_baseline_logistic(
    X_train_c, X_test_c, y_train_c, y_test_c
)

rf_credit, _ = model_trainer_credit.train_random_forest(
    X_train_c, X_test_c, y_train_c, y_test_c
)

xgb_credit, _ = model_trainer_credit.train_xgboost(
    X_train_c, X_test_c, y_train_c, y_test_c
)

# Save credit models
model_trainer_credit.save_model(logistic_credit, 'logistic_regression_credit')
model_trainer_credit.save_model(rf_credit, 'random_forest_credit')
model_trainer_credit.save_model(xgb_credit, 'xgboost_credit')

### 8. BUSINESS-ORIENTED ANALYSIS

In [ ]:
print("\n" + "="*70)
print("8. BUSINESS IMPACT ANALYSIS")
print("="*70)

from src.evaluation import ModelEvaluator

def analyze_business_impact(model, X_test, y_test, model_name):
    """Analyze business impact of model predictions."""
    evaluator = ModelEvaluator(model, X_test, y_test)
    results = evaluator.evaluate(model_name)
    
    # Cost-benefit analysis
    tn, fp, fn, tp = results['confusion_matrix'].ravel()
    
    print(f"\nBusiness Impact Analysis for {model_name}:")
    print("-" * 40)
    
    # Example costs (adjust based on business context)
    cost_fp = 50  # Cost per false positive (customer service, manual review)
    cost_fn = 500  # Cost per false negative (actual fraud loss)
    benefit_tp = 500  # Benefit per true positive (fraud prevented)
    
    total_cost = (fp * cost_fp) + (fn * cost_fn) - (tp * benefit_tp)
    cost_per_transaction = total_cost / len(y_test)
    
    print(f"False Positives: {fp} × ${cost_fp} = ${fp * cost_fp:,}")
    print(f"False Negatives: {fn} × ${cost_fn} = ${fn * cost_fn:,}")
    print(f"True Positives: {tp} × ${benefit_tp} = ${tp * benefit_tp:,}")
    print(f"Net Cost: ${total_cost:,}")
    print(f"Cost per transaction: ${cost_per_transaction:.2f}")
    
    return total_cost

# Analyze for e-commerce models
print("\nE-COMMERCE MODELS BUSINESS IMPACT:")
print("-" * 50)

cost_logistic = analyze_business_impact(logistic_model, X_test, y_test, "Logistic Regression")
cost_rf = analyze_business_impact(rf_model, X_test, y_test, "Random Forest")
cost_xgb = analyze_business_impact(xgb_model, X_test, y_test, "XGBoost")

### 9. HYPERPARAMETER TUNING

In [ ]:
print("\n" + "="*70)
print("9. HYPERPARAMETER TUNING (OPTIONAL)")
print("="*70)

# This section can be expanded with GridSearchCV or RandomizedSearchCV
# For now, we'll show a simple example

from sklearn.model_selection import GridSearchCV

def tune_random_forest(X_train, y_train):
    """Simple hyperparameter tuning for Random Forest."""
    print("Tuning Random Forest hyperparameters...")
    
    param_grid = {
        'n_estimators': [50, 100, 150],
        'max_depth': [5, 10, 15, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }
    
    rf = RandomForestClassifier(class_weight='balanced_subsample', 
                               random_state=42, n_jobs=-1)
    
    # Use AUC-PR as scoring for imbalanced data
    grid_search = GridSearchCV(
        rf, param_grid, 
        cv=3, 
        scoring='average_precision',  # AUC-PR
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best AUC-PR score: {grid_search.best_score_:.4f}")
    
    return grid_search.best_estimator_

# Uncomment to run tuning
# best_rf = tune_random_forest(X_train, y_train)

print("\n" + "="*70)
print("TASK 2 COMPLETED SUCCESSFULLY!")
print("="*70)
print("\n✅ Models trained and evaluated")
print("✅ Models saved to 'models/' directory")
print("✅ Evaluation plots saved to 'reports/figures/model_performance/'")
print("✅ Business impact analysis completed")